In [ ]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [ ]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# custom
from utils import *
import _run_constants as rc

# LOAD DATA

In [ ]:
word_df, word_id_list, word_byte_list, word_byte_array, word_byte_to_word_dict = load_input_data()

In [ ]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

## EXAMPLES OF BYTE COMPARISONS

In [ ]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [ ]:
# no letters in common
w1b & w2b

In [ ]:
# letters in common
w1b & w3b

In [ ]:
# bitwise or
w1b | w2b

In [ ]:
# this is the same as directly above
byte_encode_words('abhorcleft')

In [ ]:
byte_encode_words(ascii_lowercase)

# BUILD LEVEL 2

In [ ]:
l2_list = np.full(shape = (10000000, 3), fill_value = -1, dtype = np.int32)
row_index = 0
found_values = set()
for w1_be, w2_be in combinations(word_byte_list, 2):
    if w1_be & w2_be == 0:   
        # they share no letters in common
        l2 = w1_be | w2_be                          
        
        l2_list[row_index, :] = np.array([w1_be, w2_be, l2], dtype = np.int32)
        found_values.add(l2)
        row_index += 1

# trim the data frame
l2_list = l2_list[:row_index, :]
print(l2_list.shape)
l2_df = pd.DataFrame(data = l2_list, columns = ['w1b', 'w2b', 'l2'])

In [ ]:
l2_df['l2'].unique().shape

# BUILD LEVELS 3 THROUGH 5

In [ ]:
# so, now, let's try computing all possible pairs
total_output = np.full(shape = (1000000, 9), fill_value= -1, dtype = np.int32)
row_index = 0
for i_row, row in l2_df.iterrows():    
    # levels 1 and 2
    w1b, w2b, l2 = row

    # indexer for l3
    positional_idx_l3 = (word_byte_array & l2) == 0

    # l3 words with different letters
    output_array_w3b = word_byte_array[positional_idx_l3]    

    # l3 accumulated letters
    output_array_l3 = output_array_w3b | l2

    ## enumerate level 3
    for w3b, l3 in zip(output_array_w3b, output_array_l3):

        # build level 4

        # l4 idx
        positional_idx_l4 = (word_byte_array & l3) == 0

        # words with different letters
        output_array_w4b = word_byte_array[positional_idx_l4]    
        
        # accumulated letters
        output_array_l4 = output_array_w4b | l3

        ## enumerate level 5
        for w4b, l4 in zip(output_array_w4b, output_array_l4):

            # build level 5

            # l5 idx
            positional_idx_l5 = (word_byte_array & l4) == 0
            
            # words with different letters
            output_array_w5b = word_byte_array[positional_idx_l5]    

            if output_array_w5b.size > 0:
                    
                # accumulated letters
                output_array_l5 = output_array_w5b | l4

                ## gather and combine the output
                for w5b, l5 in zip(output_array_w5b, output_array_l5):

                    temp_list = np.array([w1b, w2b, w3b, w4b, w5b, l2, l3, l4, l5], dtype = np.int32)
                    total_output[row_index, :] = temp_list                   
                    row_index += 1


    if i_row % 10000 == 0:
        # print the number of l2 iterations and the shape of the output
        print(i_row, row_index)
    


# CREATE AND SAVE OUTPUT

In [ ]:
total_output = total_output[:row_index, :]
col_names = ['w1b', 'w2b', 'w3b', 'w4b', 'w5b', 'l2', 'l3', 'l4', 'l5']
l5_df = pd.DataFrame(data = total_output, columns = col_names)


In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.tail()

In [ ]:
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)
